# Hotel Booking
In this notebook, we will try to investigate the data and understand the business logic behind it.

## Import

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

In [109]:
df = pd.read_csv('../data/raw/hotel_bookings.csv')

In [110]:
print(f"Dataset loaded with: {df.shape[0]:,} records, {df.shape[1]} columns")

Dataset loaded with: 119,390 records, 32 columns


overwview of the fist few records

In [111]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [112]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

checking for any nulls or missing values

In [113]:
df.isnull().sum()[df.isnull().sum() > 0]

children         4
country        488
agent        16340
company     112593
dtype: int64

* Only four columns have null.
* the main columns such as (hotel, is_canceled, lead_time, adults, deposit_type, reserverd_room etc ...) don't have any null which is good sign. 
* However, the company and agent columns have nulls but they seem to be id columns (Needs further check)
* country needs further check to decide the right the action to be done!

We will check for any data type issues, to do so we will split the columns into categorical and numerical columns lists

In [79]:
# Spliting numerical and categorical columns to check if there are type issues and for further analysis
numerical_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(len(numerical_cols), len(categorical_cols), len(df.columns))


18 12 30


C:\Users\msipu\AppData\Local\Temp\ipykernel_21156\1530080434.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [76]:
categorical_cols

['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'reservation_status',
 'reservation_status_date']

In [77]:
numerical_cols

['is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests']

* Only reservation_status_date column is of type string and need to be converted. Otherwise, everything else looks fine. 

## Quick Data Overview

In this section, from the stats table we will try to get a quick overview from the data as well the averages, min/max values, check for outliers, data quality errors etc ...

Then we will try to check the vlaues of each columns to understand there use!

In [33]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


Quick overview from the stats table above and based on the averages and values we can see that:

* the average lead time is 104 days, with a minimum of 0 days and a maximum of 737 days. 
* avg number of adults per booking is 1.8, with a minimum of 0 **(why Zero?? needs further checking)** and a maximum of 20 (group bookings?).
* avg.number of children per booking is 0.1, with a minimum of 0 and a maximum of 10 (needs further check).
* avvg number of babies per booking is 0, with a minimum of 0 and a maximum of 4.
* avg number of nights per booking is 2.5, with a minimum of 0 and a maximum of 357 (needs further check).
* avg total number of special requests per booking is 0.3, with a minimum of 0 and a maximum of 5 (needs further check, what type of requests?).
* avg arrival week is 27-28 (june/july) summer season. 
* avg adr is 101 while min -6.3 and max 5400 (possibile outliers or errors)
* min-max year (2015 to 2017), 3 years of data
* booking changes and days in waiting list columns need double check (no quantiles)

Next, we will try to check columns unique values

In [26]:
def print_value_counts(df, columns, normalize=False, dropna=False):
    """Print value_counts() for each column in `columns`."""
    for col in columns:
        print(f"=== {col} ===")
        print(df[col].value_counts(normalize=normalize, dropna=dropna))
        print()

We will start by the categorical columns

In [80]:
# Calling the helper function to print value counts for categorical columns
print_value_counts(df, categorical_cols, normalize=True)

=== hotel ===
hotel
City Hotel      0.664461
Resort Hotel    0.335539
Name: proportion, dtype: float64

=== arrival_date_month ===
arrival_date_month
August       0.116233
July         0.106047
May          0.098760
October      0.093475
April        0.092880
June         0.091624
September    0.088014
March        0.082034
February     0.067577
November     0.056906
December     0.056789
January      0.049661
Name: proportion, dtype: float64

=== meal ===
meal
BB           0.773180
HB           0.121141
SC           0.089203
Undefined    0.009791
FB           0.006684
Name: proportion, dtype: float64

=== country ===
country
PRT    0.406986
GBR    0.101591
FRA    0.087235
ESP    0.071765
DEU    0.061035
         ...   
NCL    0.000008
KIR    0.000008
SDN    0.000008
ATF    0.000008
SLE    0.000008
Name: proportion, Length: 178, dtype: float64

=== market_segment ===
market_segment
Online TA        0.473046
Offline TA/TO    0.202856
Groups           0.165935
Direct           0.105587
C

## data quality issues

in previous section we found few intersting insights regarding the data so we will try to investigate more now!

### Zero guests

To validate the booking we need to have at least 1 adults per booking, but as we found earlier in the stats table (zeros for adults, childern and babies) which is not correct.

We will drop those rows!

In [89]:
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

In [91]:
# checking this case: ((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))
zero_total_guests_count = (df["total_guests"] == 0).sum()
print(f"Bookings with no guests (adults, children, babies all zero): {zero_total_guests_count}")

Bookings with no guests (adults, children, babies all zero): 180


Next, some bookings have childern and babies but no adults associated. 

In [95]:
adult_zero_count = ((df["adults"] == 0) & (df["children"] != 0) | (df["babies"] != 0)).sum()
print(f"Bookinngs with children but adults == 0: {adult_zero_count}")

Bookinngs with children but adults == 0: 1137


In [48]:
adult_zero_count = ((df["adults"] == 0)).sum()
print(f"Bookings with adults == 0: {adult_zero_count}")

Bookings with adults == 0: 403


In [93]:
# Total nights
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
total_nights_zero_count = (df["total_nights"] == 0).sum()
print(f"Bookings with total nights == 0: {total_nights_zero_count}")

Bookings with total nights == 0: 715


So, we have some booking that have zeros nights. It could be good indicator the cancellation

### Duplicate rows (booking)

In [49]:
duplicate_count = df.duplicated().sum()
duplicate_rate = duplicate_count / df.shape[0]

print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate rate: {duplicate_rate:.2%}")

Duplicate rows: 31994
Duplicate rate: 26.80%


We have 27% of duplicate rows but the client confirmed that each row in this dataset represents a unique booking. But we need to flag and mention this anyways!

### Nulls

We have few columns that have nulls, we will analyse them separately.

For the country, the safest choice is the impute the nulls with Unkown

In [60]:
df.country.isnull().sum() / df.shape[0]

np.float64(0.004087444509590418)

For company and agent, they have 94% and 13% of nulls repecetively. So, instead of droping them we will use the business knowledge to create features from them like has_agent, has_company

In [83]:
df.company.isnull().sum() / df.shape[0]

np.float64(0.943068933746545)

In [84]:
df.agent.isnull().sum() / df.shape[0]

np.float64(0.13686238378423654)